# EIT Coupled 2nd-Order ODEs — Comprehensive Notebook

Classical analog of **Electromagnetically Induced Transparency (EIT)** via two coupled harmonic oscillators with asymmetric damping.

**System:**
- Oscillator 1 (probe): strongly damped — ζ₁ (broad linewidth)
- Oscillator 2 (control): weakly damped — ζ₂ ≪ ζ₁ (narrow linewidth)
- Coupling: Ω_r² position coupling

**RWA note:** The probe drive uses `F·exp(-iω_s·t)` (complex exponential), valid under the Rotating Wave Approximation (ζ₁/ω₀ ≈ 2% — near-resonant, weak damping).

Reference: http://dx.doi.org/10.13140/RG.2.2.33199.00164/1

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.integrate import solve_ivp
from scipy.fftpack import fft, fftfreq
from scipy.signal.windows import hann

# Portable output directory (no hardcoded paths)
FIG_DIR = Path("figures")
FIG_DIR.mkdir(exist_ok=True)

# ── Physical parameters (SI-normalised units) ─────────────────────────────
OMEGA0  = 2.0      # rad/s  — natural frequency (shared by both oscillators)
ZETA1   = 4.0e-2   # rad/s  — probe oscillator damping (broad linewidth)
ZETA2   = 1.0e-6   # rad/s  — control oscillator damping (narrow linewidth)
F_PROBE = 0.1      # a.u.   — probe force amplitude
OMEGA_S = 0.0      # rad/s  — probe detuning (0 = on resonance)
OMEGA_R = 0.1      # rad/s  — coupling (Rabi) frequency

ATOL = RTOL = 1e-10  # DOP853 solver tolerances


## Section 1 — Single Damped Oscillator

Equation of motion: **ẍ + γẋ + ω₀²x = 0**

Four regimes depending on γ vs ω₀:
| Regime | Condition |
|---|---|
| Overdamped | γ > ω₀ |
| Critically damped | γ = ω₀ |
| Underdamped | γ < ω₀ |
| Undamped | γ = 0 |

In [ ]:
def solve_oscillator(gamma, omega0, ts=6.0, n_pts=100_000, x0=1.0, dx0=0.0):
    """Solve ẍ + γẋ + ω₀²x = 0 numerically.

    Parameters
    ----------
    gamma  : damping coefficient (rad/s)
    omega0 : natural frequency (rad/s)
    ts     : simulation duration (s)
    """
    def odes(t, S):
        x, v = S
        return [v, -gamma * v - omega0**2 * x]  # fixed: was -omega0*x (missing **2)

    t = np.linspace(0, ts, n_pts)
    sol = solve_ivp(odes, (0, ts), [x0, dx0],
                    method='DOP853', t_eval=t, atol=ATOL, rtol=RTOL)
    assert sol.success, f"Solver failed: {sol.message}"
    return sol


In [ ]:
regimes = [
    ("Overdamped  γ > ω₀",  1.5),
    ("Critical    γ = ω₀",  1.0),
    ("Underdamped γ < ω₀",  0.3),
    ("Undamped    γ = 0",   0.0),
]

plt.figure(figsize=(8, 4))
for label, gamma in regimes:
    sol = solve_oscillator(gamma=gamma * 2 * np.pi, omega0=1.0 * 2 * np.pi)
    plt.plot(sol.t, np.real(sol.y[0]), label=label)

plt.xlabel("t (s)")
plt.ylabel("x(t)")
plt.title("Damped Harmonic Oscillator — Four Regimes")
plt.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / "damped_regimes.pdf")
plt.show()


In [ ]:
# Undamped case has exact solution x(t) = x0·cos(ω₀·t)
omega0_test = 1.0
sol_undamped = solve_oscillator(gamma=0.0, omega0=omega0_test, ts=6.0)
x_analytical = np.cos(omega0_test * sol_undamped.t)

max_err = np.max(np.abs(np.real(sol_undamped.y[0]) - x_analytical))
print(f"Max error vs analytical cos(ω₀t): {max_err:.2e}  (expect < 1e-8)")


In [ ]:
# For undamped SHO, total energy E = ½v² + ½ω₀²x² must be constant
omega0_test = 1.0
x = np.real(sol_undamped.y[0])
v = np.real(sol_undamped.y[1])
E = 0.5 * v**2 + 0.5 * omega0_test**2 * x**2
drift = (E.max() - E.min()) / E.mean()
print(f"Relative energy drift: {drift:.2e}  (expect < 1e-8)")

plt.figure(figsize=(8, 3))
plt.plot(sol_undamped.t, E)
plt.xlabel("t (s)"); plt.ylabel("E (a.u.)")
plt.title("Energy Conservation — Undamped Oscillator")
plt.tight_layout(); plt.show()


## Section 2 — Driven Oscillator

Equation of motion: **ẍ + 2γẋ + ω₀²x = ω₀²·x_drive·cos(ω_d·t)**

Using a real cosine drive. Hann window applied to FFT to reduce spectral leakage.

In [ ]:
def solve_driven(gamma, omega0, omega_d, x_drive=1e-4, ts=100.0, n_pts=100_000):
    """Solve ẍ + 2γẋ + ω₀²x = ω₀²·x_drive·cos(ω_d·t)."""
    def odes(t, S):
        x, v = S
        return [v, -2*gamma*v - omega0**2*x + omega0**2*x_drive*np.cos(omega_d*t)]
    t = np.linspace(0, ts, n_pts)
    sol = solve_ivp(odes, (0, ts), [1.0, 0.0],
                    method='DOP853', t_eval=t, atol=ATOL, rtol=RTOL)
    assert sol.success, sol.message
    return sol

omega0_drv = 10 * 2 * np.pi
plt.figure(figsize=(8, 4))
for gf in [0.1, 0.3, 0.5, 0.7, 0.9]:
    gamma = gf * 2 * np.pi
    sol = solve_driven(gamma, omega0_drv, omega0_drv)
    N = len(sol.t); T = 100.0
    freq = 2 * np.pi * np.arange(N) / T
    win = hann(N)
    X = fft(np.real(sol.y[0]) * win)
    plt.plot(freq / omega0_drv, np.real(X) / N / 1e-4, label=f"γ/ω₀={gf}")

plt.xlim(0, 2)
plt.xlabel("ω / ω₀"); plt.ylabel("Re[x̃(ω)] / x_drive")
plt.title("Driven Oscillator — Frequency Response (Hann-windowed)")
plt.legend(); plt.tight_layout()
plt.savefig(FIG_DIR / "driven_response.pdf"); plt.show()


## Section 3 — Analytical EIT Susceptibility

Solving the coupled equations analytically in the frequency domain (RWA) gives:

**χ(ω_s) = −2πiF·ω_s·(ω₀²−ω_s²−iζ₂ω_s) / [(ω₀²−ω_s²−iζ₁ω_s)(ω₀²−ω_s²−iζ₂ω_s) − Ω_r⁴]**

- Re[χ] → probe absorption
- −Im[χ] → dispersion
- EIT dip appears at ω_s = ω₀ when Ω_r > 0 (coupling opens transparency window)

In [ ]:
def susceptibility(omega_s, Omega_r,
                   omega0=OMEGA0, zeta1=ZETA1, zeta2=ZETA2, F=F_PROBE):
    """Analytical complex susceptibility of the coupled oscillator EIT system.

    Parameters
    ----------
    omega_s : probe frequency array (rad/s)
    Omega_r : coupling (Rabi) frequency (rad/s)

    Returns
    -------
    chi : complex array — Re→absorption, -Im→dispersion
    """
    delta = omega0**2 - omega_s**2
    D1 = delta - 1j * zeta1 * omega_s   # probe oscillator denominator
    D2 = delta - 1j * zeta2 * omega_s   # control oscillator denominator
    return (-2 * np.pi * 1j * F * omega_s * D2) / (D1 * D2 - Omega_r**4)


In [ ]:
omega_s = np.linspace(OMEGA0 - 1.5, OMEGA0 + 1.5, 500_000)
detuning = omega_s - OMEGA0

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8, 6), sharex=True)
colors = plt.cm.viridis(np.linspace(0, 0.9, 6))

for i, Or in enumerate([0.0, 0.1, 0.2, 0.3, 0.4, 0.5]):
    chi = susceptibility(omega_s, Or)
    ax1.plot(detuning, np.real(chi),  color=colors[i], label=f"Ω_r={Or}")
    ax2.plot(detuning, -np.imag(chi), color=colors[i], label=f"Ω_r={Or}")

ax1.set_ylabel("Re[χ] — Absorption"); ax1.set_xlim(-0.4, 0.4); ax1.legend(fontsize=8)
ax2.set_ylabel("−Im[χ] — Dispersion"); ax2.set_xlabel("Detuning ω_s − ω₀ (rad/s)")
fig.suptitle("EIT Susceptibility vs Coupling Strength Ω_r")
plt.tight_layout(); plt.savefig(FIG_DIR / "eit_susceptibility.pdf"); plt.show()


In [ ]:
# EIT linewidth should scale as Ω_r² / Γ  (Γ = ζ₁ = broad linewidth)
Omega_r_vals = np.linspace(0.05, 0.5, 25)
linewidths = []

for Or in Omega_r_vals:
    chi = susceptibility(omega_s, Or)
    absorption = np.real(chi)
    idx_res = np.argmin(np.abs(omega_s - OMEGA0))
    window_half = 20_000
    local = absorption[idx_res - window_half : idx_res + window_half]
    local_freq = omega_s[idx_res - window_half : idx_res + window_half]
    peak = local.max()
    dip  = absorption[idx_res]
    half = (peak + dip) / 2
    above = local_freq[local < half]
    linewidths.append(above[-1] - above[0] if len(above) > 1 else np.nan)

predicted = Omega_r_vals**2 / ZETA1

plt.figure(figsize=(6, 4))
plt.plot(Omega_r_vals**2, linewidths, 'o', label="Measured FWHM")
plt.plot(Omega_r_vals**2, predicted,  '--', label="Ω_r²/Γ  (theory)")
plt.xlabel("Ω_r²  (rad²/s²)"); plt.ylabel("EIT linewidth (rad/s)")
plt.title("EIT Linewidth Scaling: Ω_r²/Γ"); plt.legend()
plt.tight_layout(); plt.savefig(FIG_DIR / "eit_linewidth_scaling.pdf"); plt.show()


In [ ]:
# Group index n_g = 1 + (ω₀/2) · dRe[χ]/dω  evaluated at ω = ω₀
chi = susceptibility(omega_s, OMEGA_R)
dchi_dw = np.gradient(np.real(chi), omega_s[1] - omega_s[0])
idx0 = np.argmin(np.abs(omega_s - OMEGA0))
n_g = 1 + (OMEGA0 / 2) * dchi_dw[idx0]
print(f"Coupling Ω_r = {OMEGA_R}  →  Group index n_g ≈ {n_g:.0f}")
print(f"Light slowed by factor {n_g:.0f}× compared to free propagation")

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(omega_s - OMEGA0, np.real(chi),  label="Re[χ] absorption")
axes[0].plot(omega_s - OMEGA0, -np.imag(chi), label="−Im[χ] dispersion")
axes[0].set_xlim(-0.3, 0.3); axes[0].legend(); axes[0].set_xlabel("Detuning (rad/s)")
axes[0].set_title("EIT Susceptibility")

axes[1].plot(omega_s - OMEGA0, dchi_dw)
axes[1].axvline(0, ls='--', c='k', alpha=0.4)
axes[1].set_xlim(-0.3, 0.3); axes[1].set_xlabel("Detuning (rad/s)")
axes[1].set_ylabel("dRe[χ]/dω"); axes[1].set_title(f"Dispersion Slope  (n_g ≈ {n_g:.0f})")
plt.tight_layout(); plt.savefig(FIG_DIR / "group_velocity.pdf"); plt.show()


## Section 4 — Coupled ODE System (Numerical Solver)

Equations of motion (4 first-order ODEs, RWA):

| Variable | Equation |
|---|---|
| ẋ₁ = v₁ | |
| v̇₁ | −ζ₁v₁ − ω₀²x₁ + Ω_r²x₂ + F·e^{−iω_s t} |
| ẋ₂ = v₂ | |
| v̇₂ | −ζ₂v₂ − ω₀²x₂ + Ω_r²x₁ |

**Transient removal:** The damping time τ₁ = 1/ζ₁ ≈ 25 s. Only t > 5τ₁ ≈ 125 s is used for FFT analysis, ensuring steady-state response.  
**Windowing:** Hann window applied to eliminate spectral leakage.

In [ ]:
def solve_coupled_eit(Omega_r, omega0=OMEGA0, zeta1=ZETA1, zeta2=ZETA2,
                      F=F_PROBE, omega_s=OMEGA_S,
                      ts=2000.0, n_pts=1_000_000, atol=ATOL, rtol=RTOL):
    """Solve coupled EIT oscillator system numerically (RWA).

    State vector: S = [x₁, v₁, x₂, v₂]

    Parameters
    ----------
    Omega_r : coupling frequency (rad/s)
    ts      : total simulation time (s) — should be >> 5/zeta1 for steady state
    n_pts   : number of time evaluation points
    """
    def odes(t, S):
        x1, v1, x2, v2 = S
        return [
            v1,
            -zeta1*v1 - omega0**2*x1 + Omega_r**2*x2 + F*np.exp(-1j*omega_s*t),
            v2,
            -zeta2*v2 - omega0**2*x2 + Omega_r**2*x1,
        ]

    t_eval = np.linspace(0, ts, n_pts)
    S0 = np.zeros(4, dtype=np.complex128)
    sol = solve_ivp(odes, (0, ts), S0,
                    method='DOP853', t_eval=t_eval, atol=atol, rtol=rtol)
    assert sol.success, f"ODE solver failed: {sol.message}"
    return sol


In [ ]:
print("Solving coupled EIT system (this may take ~1 min)...")
sol = solve_coupled_eit(OMEGA_R, ts=2000.0, n_pts=1_000_000)
print("Done.")

fig, axes = plt.subplots(2, 1, figsize=(10, 5), sharex=True)
axes[0].plot(sol.t, np.real(sol.y[0]), lw=0.5, label="x₁ probe")
axes[0].plot(sol.t, np.real(sol.y[2]), lw=0.5, label="x₂ control")
axes[0].set_ylabel("Position"); axes[0].legend()
axes[1].plot(sol.t, np.real(sol.y[1]), lw=0.5, label="v₁ = ẋ₁")
axes[1].plot(sol.t, np.real(sol.y[3]), lw=0.5, label="v₂ = ẋ₂")
axes[1].set_ylabel("Velocity"); axes[1].set_xlabel("t (s)"); axes[1].legend()
fig.suptitle("Coupled EIT Oscillators — Time Domain")
plt.tight_layout(); plt.savefig(FIG_DIR / "time_domain.pdf"); plt.show()


In [ ]:
# Discard transient: keep only t > 5 × damping_time
tau = 1.0 / ZETA1          # ≈ 25 s
t_cut = 5 * tau             # ≈ 125 s
mask = sol.t >= t_cut

t_ss  = sol.t[mask]
v1_ss = np.real(sol.y[1, mask])   # probe velocity (steady-state)

N_ss   = len(t_ss)
T_ss   = t_ss[-1] - t_ss[0]
win    = hann(N_ss)
freq_ss = fftfreq(N_ss, d=T_ss / N_ss) * 2 * np.pi

X_re = np.real(fft(v1_ss * win)) / N_ss
X_im = np.imag(fft(v1_ss * win)) / N_ss

plt.figure(figsize=(8, 4))
plt.plot(freq_ss - OMEGA0, X_re, label="Re[FFT(v₁)]  ~ absorption")
plt.plot(freq_ss - OMEGA0, X_im, label="Im[FFT(v₁)]  ~ dispersion")
plt.xlim(-0.5, 0.5)
plt.xlabel("Detuning ω − ω₀ (rad/s)"); plt.ylabel("Normalised FFT amplitude")
plt.title("Steady-State EIT Response (Hann-windowed, transient removed)")
plt.legend(); plt.tight_layout()
plt.savefig(FIG_DIR / "eit_numerical_fft.pdf"); plt.show()


## Section 5 — Normal Mode Analysis (Bright / Dark States)

The two oscillators can be decomposed into **normal modes**:

| Mode | Definition | Property |
|---|---|---|
| Bright x₊ = (x₁+x₂)/√2 | Couples to probe field | Absorbs — shows double peak |
| Dark   x₋ = (x₁−x₂)/√2 | Decoupled from probe | Transparent — near-zero at resonance |

Destructive interference in the dark mode is the classical origin of the EIT transparency window.

In [ ]:
x1 = np.real(sol.y[0, mask])
x2 = np.real(sol.y[2, mask])

x_bright = (x1 + x2) / np.sqrt(2)
x_dark   = (x1 - x2) / np.sqrt(2)

X_bright = np.abs(fft(x_bright * win))
X_dark   = np.abs(fft(x_dark   * win))

plt.figure(figsize=(8, 4))
plt.plot(freq_ss - OMEGA0, X_bright / X_bright.max(), label="Bright mode x₊")
plt.plot(freq_ss - OMEGA0, X_dark   / X_bright.max(), label="Dark mode x₋")
plt.xlim(-0.5, 0.5)
plt.xlabel("Detuning ω − ω₀ (rad/s)"); plt.ylabel("|FFT| (normalised)")
plt.title("Normal Mode Decomposition: Bright vs Dark State")
plt.legend(); plt.tight_layout()
plt.savefig(FIG_DIR / "bright_dark_modes.pdf"); plt.show()


## Section 6 — Convergence Study

Verify that the choice of `atol=rtol=1e-10` produces results independent of tolerance.

In [ ]:
tol_list = [1e-8, 1e-10, 1e-12]
plt.figure(figsize=(8, 4))

for tol in tol_list:
    s = solve_coupled_eit(OMEGA_R, ts=500.0, n_pts=200_000, atol=tol, rtol=tol)
    m = s.t >= 5 / ZETA1
    N = m.sum(); T = s.t[m][-1] - s.t[m][0]
    w = hann(N); f = fftfreq(N, d=T/N) * 2 * np.pi
    X = np.real(fft(np.real(s.y[1, m]) * w))
    X_n = X / np.max(np.abs(X))
    plt.plot(f - OMEGA0, X_n, label=f"atol=rtol={tol:.0e}", lw=1.5 if tol==1e-10 else 1)

plt.xlim(-0.5, 0.5)
plt.xlabel("Detuning (rad/s)"); plt.ylabel("Normalised Re[FFT]")
plt.title("Convergence: Three DOP853 Tolerances (curves should overlap)")
plt.legend(); plt.tight_layout()
plt.savefig(FIG_DIR / "convergence.pdf"); plt.show()


## Section 7 — Analytical vs Numerical Comparison

Overlay the ODE FFT result with the analytical susceptibility formula on the same frequency axis.

In [ ]:
# Analytical on the same frequency grid as the FFT
chi_theory = susceptibility(freq_ss + OMEGA0, OMEGA_R)
chi_re = np.real(chi_theory)

# Normalise both to their peak for fair shape comparison
X_norm   = X_re / np.max(np.abs(X_re))
chi_norm = chi_re / np.max(np.abs(chi_re))

plt.figure(figsize=(8, 4))
plt.plot(freq_ss - OMEGA0, X_norm,   label="Numerical ODE (FFT)")
plt.plot(freq_ss - OMEGA0, chi_norm, '--', lw=2, label="Analytical χ(ω)")
plt.xlim(-0.5, 0.5)
plt.xlabel("Detuning ω − ω₀ (rad/s)"); plt.ylabel("Normalised amplitude")
plt.title("Analytical vs Numerical EIT Susceptibility")
plt.legend(); plt.tight_layout()
plt.savefig(FIG_DIR / "analytical_vs_numerical.pdf"); plt.show()

# Quantitative residual
rms = np.sqrt(np.nanmean((X_norm - chi_norm)**2))
print(f"RMS residual (normalised): {rms:.4f}")


## Summary

This notebook fixes all critical bugs from the original 5 notebooks and adds:
- Analytical verification against `cos(ω₀t)`
- Energy conservation check
- EIT linewidth extraction and Ω_r²/Γ scaling verification
- Group velocity / slow light calculation
- Bright/dark normal mode decomposition
- Convergence study (3 tolerances)
- Quantitative analytical vs numerical comparison

All figures saved to `figures/`.